### Covariance Matrix Estimation
March 2025

*Imports*

In [44]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LinearRegression
from datetime import date
import plotly.graph_objects as go

**Data**

*Downloading*

In [2]:
# SP500 Tickers
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
df = pd.read_html(url)[0]
df = df[['Symbol', 'Security']]

# Returns
tickers = list(df['Symbol'].unique())
market_ticker = '^GSPC'

start_date = date(2022,1,1)
end_date = date(2025,1,1)
data = yf.download(tickers + [market_ticker], start=start_date, end=end_date)['Close']

[                       0%                       ]

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  504 of 504 completed

2 Failed downloads:
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (1d 2022-01-01 -> 2025-01-01)')
['BRK.B']: YFTzMissingError('possibly delisted; no timezone found')


*Cleaning*

In [34]:
data = data.dropna(axis = 1)
stocks_data = data.loc[:, data.columns.difference(['^GSPC'])]
market_data = data[market_ticker]

# Calculate daily returns
stocks_returns = stocks_data.pct_change().dropna()
market_returns = market_data.pct_change().dropna()

# Risk-free rate (2% annual, approx. 0.02/252 daily)
rf_rate = 0.02 / 252
market_data = data[market_ticker]

# Calculate daily returns
stocks_returns = stocks_data.pct_change().dropna()
market_returns = market_data.pct_change().dropna()

# Risk-free rate
rf_rate = 0.02 / 252

#### **Estimating Covariance Matrix**

In [35]:
sample_returns = stocks_returns[:-252]
market_sample_returns = market_returns[:-252]

*Sample Covariance Matrix*

In [36]:
sample_cov = sample_returns.cov()

*Rolling Window*

In [37]:
last_252_returns = sample_returns.tail(252)
window_cov = last_252_returns.cov()

*Exponentially Weighted Covariance*

In [38]:
ewc_cov = pd.DataFrame(index=sample_returns.columns, columns=sample_returns.columns)
for stock1 in sample_returns.columns:
    for stock2 in sample_returns.columns:
        ewc_cov.loc[stock1, stock2] = sample_returns[stock1].ewm(span=252, adjust=False).cov(sample_returns[stock2]).iloc[-1]

ewc_cov = ewc_cov.astype(np.float64)

*Shrinkage*

In [39]:
lw = LedoitWolf()
lw.fit(sample_returns)
shrinkage_cov = lw.covariance_

#### **Mean Variance Optimization**

*Expected Returns*

In [40]:
# Calculate the expected returns using CAPM: R = Rf + Beta * (Rm - Rf)
betas = []
for stock in sample_returns.columns:
    X = market_sample_returns.values.reshape(-1, 1)
    y = sample_returns[stock].values 
    
    model = LinearRegression()
    model.fit(X, y)
    betas.append(model.coef_[0])

market_return = market_sample_returns.mean()
expected_returns = rf_rate + np.array(betas) * (market_return - rf_rate)

*Mean Variance Optimization*

In [ ]:
# Optimal weights under MVO
def mean_variance_optimization(cov_matrix, expected_returns):
    ones = np.ones(len(expected_returns))
    inv_cov = np.linalg.inv(cov_matrix)
    w_opt = np.dot(inv_cov, expected_returns) / np.dot(ones, np.dot(inv_cov, expected_returns))
    return w_opt

In [42]:
# Weights under each method
weights_sample = mean_variance_optimization(sample_cov, expected_returns)
weights_ewc = mean_variance_optimization(ewc_cov, expected_returns)
weights_shrinkage = mean_variance_optimization(shrinkage_cov, expected_returns)
weights_rolling = mean_variance_optimization(window_cov, expected_returns)


# Forward Portfolio Returns
def simulate_portfolio_performance(weights, returns):
    portfolio_returns = np.dot(returns.values, weights)
    cumulative_returns = (1 + portfolio_returns).cumprod() - 1
    return cumulative_returns

# Simulate performance for each portfolio
performance_sample = simulate_portfolio_performance(weights_sample, stocks_returns[-252:])
performance_ewc = simulate_portfolio_performance(weights_ewc, stocks_returns[-252:])
performance_shrinkage = simulate_portfolio_performance(weights_shrinkage, stocks_returns[-252:])
performance_rolling = simulate_portfolio_performance(weights_rolling, stocks_returns[-252:])

*Plot*

In [52]:
fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x = stocks_returns[-252:].index,
        y = performance_sample,
        name = 'Sample Covariance'
    )
)


fig.add_trace(
    go.Scatter(
        x = stocks_returns[-252:].index,
        y = performance_ewc,
        name = 'Exponentially Weighted Covariance'
    )
)

fig.add_trace(
    go.Scatter(
        x = stocks_returns[-252:].index,
        y = performance_rolling,
        name = 'Rolling Window Covariance'
    )
)

fig.add_trace(
    go.Scatter(
        x = stocks_returns[-252:].index,
        y = performance_shrinkage,
        name = 'Shrinkage Covariance'
    )
)

fig.update_layout(
                  showlegend=True,
                  margin=dict(l=10, r=10, t=50, b=10),
                  legend=dict(orientation="h",yanchor="top",y=-0.1,xanchor="center",x=0.5),
                  width = 800,height = 400,
                  xaxis_title = 'Date',
                  yaxis_title = 'Cumulative Returns',
                  xaxis=dict(title_standoff=3),
                  title = 'Portfolio Performance by Covariance Estimates',
                  template = 'plotly_white'
                )

fig.show()